In [2]:
from google.colab import files
uploaded = files.upload()

Saving test-2.csv to test-2.csv
Saving train-2.csv to train-2.csv


In [17]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.9 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
train = pd.read_csv('/content/train-2.csv')
test = pd.read_csv('/content/test-2.csv')

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [7]:
train['Heart Disease'].value_counts()

,count
Heart Disease,
Absence,347546
Presence,282454


In [8]:
train['Heart Disease'] = train['Heart Disease'].map({
    'Absence': 0,
    'Presence': 1
})

In [13]:
X = train.drop(columns=['Heart Disease','id'])
y = train['Heart Disease']
test = test.drop(columns=['id'])

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [14]:
#baseline model

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [16]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_preds = log_model.predict_proba(X_val)[:, 1]
log_auc = roc_auc_score(y_val, log_preds)
print("Logistic Regression ROC-AUC:", log_auc)

Logistic Regression ROC-AUC: 0.9483396189090014


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [18]:
from catboost import CatBoostClassifier

In [19]:
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)
cat_model.fit(X_train, y_train)

0:	total: 478ms	remaining: 7m 57s
100:	total: 33.7s	remaining: 4m 59s
200:	total: 1m 1s	remaining: 4m 3s
300:	total: 1m 17s	remaining: 2m 58s
400:	total: 1m 31s	remaining: 2m 16s
500:	total: 1m 49s	remaining: 1m 49s
600:	total: 2m 5s	remaining: 1m 23s
700:	total: 2m 19s	remaining: 59.6s
800:	total: 2m 46s	remaining: 41.3s
900:	total: 3m	remaining: 19.9s
999:	total: 3m 14s	remaining: 0us


In [20]:
cat_preds = cat_model.predict_proba(X_val)[:, 1]
cat_auc = roc_auc_score(y_val, cat_preds)
print("CatBoost ROC-AUC:", cat_auc)

CatBoost ROC-AUC: 0.9560477392310069


In [22]:
from lightgbm import LGBMClassifier

In [23]:
lgb_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X_train, y_train)

lgb_preds = lgb_model.predict_proba(X_val)[:, 1]
lgb_auc = roc_auc_score(y_val, lgb_preds)

print("LightGBM ROC-AUC:", lgb_auc)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.110489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 673
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
LightGBM ROC-AUC: 0.9560120478376705


In [24]:
import pandas as pd
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'LightGBM',
        'CatBoost'
    ],
    'ROC-AUC': [
        log_auc,
        lgb_auc,
        cat_auc
    ]
})
results.sort_values(by='ROC-AUC', ascending=False)

,Model,ROC-AUC
2,CatBoost,0.956048
1,LightGBM,0.956012
0,Logistic Regression,0.948340


In [26]:
best_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)
best_model.fit(X, y)

0:	total: 375ms	remaining: 3m 7s
100:	total: 26.9s	remaining: 1m 46s
200:	total: 43.5s	remaining: 1m 4s
300:	total: 1m 1s	remaining: 40.7s
400:	total: 1m 18s	remaining: 19.3s
499:	total: 1m 35s	remaining: 0us


In [28]:
submission = pd.DataFrame({
    'id': test.index if 'id' not in test.columns else test['id'],
    'Heart Disease': test_preds
})
submission.to_csv('submission.csv', index=False)

In [29]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>